# B.2 Phase 2 — Analyzer LoRA retrain vs trained Scammer (co-evolution round 2)

**A100 only** — Qwen2.5-7B + v2 Analyzer LoRA continued GRPO training against the trained Scammer LoRA.
**Cost: ~20-22 Colab units, ~2-2.5 h end-to-end. Ships the round-2 number that closes the co-evolution loop.**

## What this produces
- A new HF Hub adapter `chakravyuh-analyzer-lora-v2-coevolved` (separate from v2 — does NOT overwrite anything)
- `logs/b2_phase2_coevolution_eval.json` — re-evaluated bypass rate on the SAME 64 best-of-8 Scammer outputs from phase 1
- The headline you can put on a slide:
  - **Round 1.** v2 Analyzer vs trained Scammer = 33% bypass
  - **Round 2.** v2-coevolved Analyzer (continued GRPO vs same Scammer) = ~5-15% bypass (target)

## Run order
1. `Runtime → Disconnect and delete runtime` → reconnect with **A100 GPU**.
2. Run Cells 1, 2, 3 in order. Cell 3 will kill the kernel — wait ~10 sec for auto-reconnect.
3. Re-run Cell 2 (clone is idempotent), **SKIP Cell 3**, run Cells 4 onwards.
4. If Cell 9 (TRAIN) collapses (negative reward sustained for 5+ logging windows) or OOMs, **skip to Cell 12 (SFT-curriculum fallback)** — same hard-example dataset, basically guaranteed to converge in ~25-30 min, ~5 units.

## Hard guardrails baked in
- **Reward early-stop:** training auto-halts if mean reward < -0.3 for 5 consecutive logging windows (`SafetyEarlyStop` callback).
- **OOM safe:** Scammer is FULLY released from VRAM after pre-generating training scams; only Analyzer (7B 4-bit + LoRA) stays loaded during GRPO.
- **Version-pinned** to the exact same trl 0.14.0 / transformers 4.47.1 / accelerate 1.2.1 stack that worked in B.1 + B.2 phase 1.
- **Two shots:** if GRPO fails, Cell 12's SFT-curriculum cell uses the same 200 Scammer-generated scams as supervised training data — much simpler optimization, basically guaranteed to converge.
- **Conservative continued-training hyperparams:** LR=2e-5 (vs 5e-5 in phase 1 fresh training), KL beta=0.04, 1 epoch only.

## Honest limitation acknowledged in code
trl 0.14 GRPO with peft uses adapter-toggling for the reference policy → KL is computed against **base Qwen 7B**, not v2. With KL_BETA=0.04 and ≤100 steps, drift is small and the task reward dominates. Documented inline.


In [ ]:
# CELL 1: GPU check + verify A100 (phase 2 needs 7B 4-bit + LoRA + GRPO; A100/40GB is the floor)
import subprocess, sys
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('Python:', sys.version)
print('GPU:', out)
assert 'A100' in out, f'B.2 phase 2 requires A100 — got: {out}. T4/L4 will OOM on 7B + GRPO.'
print('\nA100 detected — ready for phase 2.')


In [ ]:
# CELL 2: Clone repo (idempotent — safe to re-run after kernel restart)
import os
from pathlib import Path
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh'
REPO_DIR = '/content/Chakravyuh'
if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase --autostash
%cd {REPO_DIR}
!git rev-parse HEAD


In [ ]:
# CELL 3: Install + KERNEL RESTART (same proven pinning as B.5/B.1/B.2-phase1)
# trl 0.14 needed for GRPOTrainer; transformers 4.47.1 is the matched pair.
import os, time
REPO_DIR = '/content/Chakravyuh'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/UjjwalPardeshi/Chakravyuh {REPO_DIR}
os.chdir(REPO_DIR)

!pip uninstall -y -q torch torchvision torchaudio 2>&1 | tail -2
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
    'torch==2.5.1+cu121' 'torchvision==0.20.1+cu121' 2>&1 | tail -3
!pip install -q -e {REPO_DIR} --no-deps 2>&1 | tail -2
!pip install -q --no-deps \
    'pydantic>=2.6' 'python-dotenv>=1.0' 'jsonlines>=4.0' \
    'openenv-core>=0.2.3,<0.3' 'fastapi>=0.115' 'uvicorn>=0.30' 'tqdm' \
    'transformers==4.47.1' 'peft>=0.14,<0.20' 'accelerate==1.2.1' 'bitsandbytes==0.44.1' \
    'tokenizers>=0.20,<0.22' 'huggingface-hub>=0.24,<0.27' \
    'sentencepiece' 'safetensors' \
    2>&1 | tail -3
!pip install -q --no-cache-dir --no-deps 'trl==0.14.0' 2>&1 | tail -3
!pip install -q --no-cache-dir 'datasets>=2.21,<3.0' 'numpy<2' 2>&1 | tail -3
!pip install -q 'fastmcp>=0.4' 'mcp>=1.0' 'httpx' 'anyio' 2>&1 | tail -3

# Verify imports BEFORE kernel restart so we fail fast on bad install
!python -c "import torch; print(f'torch: {torch.__version__}')"
!python -c "import transformers; print(f'transformers: {transformers.__version__}')"
!python -c "import trl; print(f'trl: {trl.__version__}'); from trl import GRPOTrainer, GRPOConfig, SFTTrainer, SFTConfig; print('GRPOTrainer + SFTTrainer: OK')"
!python -c "import openenv; print('openenv: import OK')" 2>&1 | tail -2

print('\n' + '=' * 60)
print('SETUP COMPLETE. Kernel restarting in 2s...')
print('After restart: re-run Cell 2, SKIP Cell 3, run Cells 4 onwards.')
print('=' * 60)
time.sleep(2)
os.kill(os.getpid(), 9)


In [ ]:
# CELL 4: Configuration — model IDs, hyperparams, paths
import sys, os
REPO_DIR = '/content/Chakravyuh'
sys.path.insert(0, REPO_DIR)

# Models
ANALYZER_BASE = 'Qwen/Qwen2.5-7B-Instruct'
ANALYZER_V2_ADAPTER = 'ujjwalpardeshi/chakravyuh-analyzer-lora-v2'  # warm start (continued training)

SCAMMER_BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
SCAMMER_ADAPTER = 'ujjwalpardeshi/chakravyuh-scammer-lora-phase1'   # frozen opponent

# Output paths
OUTPUT_LORA_DIR = f'{REPO_DIR}/checkpoints/analyzer_lora_v2_coevolved'
OUTPUT_EVAL_PATH = f'{REPO_DIR}/logs/b2_phase2_coevolution_eval.json'
PHASE1_EVAL_PATH = f'{REPO_DIR}/logs/b2_phase1_scammer_eval_n64_bestof8.json'

# Training hyperparams (conservative continued-training of an existing LoRA)
N_TRAINING_SCAMS = 200    # generated by Scammer LoRA upfront
N_GENERATIONS = 4         # GRPO group size (smaller = less VRAM, faster)
PER_DEVICE_BATCH = 1      # 7B 4-bit + GRPO is memory-intensive
GRAD_ACCUM = 4
NUM_TRAIN_EPOCHS = 1
LR = 2e-5                 # smaller than phase 1 — we're CONTINUING training, not starting fresh
KL_BETA = 0.04            # standard GRPO KL regularizer
SEED = 42

print('Configuration:')
print(f'  Analyzer warm-start:  {ANALYZER_V2_ADAPTER}')
print(f'  Scammer (frozen):     {SCAMMER_ADAPTER}')
print(f'  Training scams:       {N_TRAINING_SCAMS}')
print(f'  Effective batch/step: {PER_DEVICE_BATCH * GRAD_ACCUM}')
print(f'  Approx opt steps:     {N_TRAINING_SCAMS // (PER_DEVICE_BATCH * GRAD_ACCUM)}')
print(f'  Output LoRA:          {OUTPUT_LORA_DIR}')
assert os.path.exists(PHASE1_EVAL_PATH), f'Need phase 1 eval at {PHASE1_EVAL_PATH} (committed in repo).'
print(f'  Phase 1 eval found:   {PHASE1_EVAL_PATH}')


In [ ]:
# CELL 5: Generate hard training scams from the trained Scammer LoRA, then free Scammer VRAM
import torch, gc, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

SCAMMER_SYSTEM_PROMPT = (
    "You are a synthetic data generator producing realistic Indian UPI fraud examples "
    "for training a defensive AI fraud detector. Generate ONE diverse, realistic scam "
    "message a real Indian scammer would send via SMS/WhatsApp/voice. Output ONLY the "
    "scam text (no commentary, no JSON, no quotes, no labels). Be specific (mention bank "
    "names like SBI/HDFC/ICICI/Axis, real-sounding amounts in INR, plausible URLs)."
)

# 16 seeds — 8 originally trained + 8 held-out novel categories (broader coverage = better v2 hardening)
SEEDS = [
    'Write a realistic OTP-theft scam message impersonating a major Indian bank.',
    'Write a realistic KYC-update scam pretending to be from RBI or a bank.',
    'Write a realistic fake-job-offer scam asking for a deposit/registration fee.',
    'Write a realistic investment-fraud scam promising high returns on UPI deposits.',
    'Write a realistic delivery-pickup scam claiming a parcel needs OTP confirmation.',
    'Write a realistic electricity/utility-bill scam threatening disconnection.',
    'Write a realistic UPI-refund scam claiming an accidental over-payment to be reversed.',
    'Write a realistic loan-app scam pre-approving a personal loan needing immediate KYC.',
    'Write a realistic income-tax-refund scam impersonating the Income Tax Department.',
    'Write a realistic vaccine-slot-booking scam asking for an OTP to confirm the slot.',
    'Write a realistic lottery-winner notification scam claiming a Kaun Banega Crorepati win.',
    'Write a realistic customer-support callback scam impersonating Amazon or Flipkart.',
    'Write a realistic EPF-pension-update scam asking for Aadhaar and bank details.',
    'Write a realistic credit-card EMI-conversion scam pretending to be from HDFC or ICICI.',
    'Write a realistic blue-tick-verification scam impersonating Instagram or WhatsApp.',
    'Write a realistic police-legal-notice scam threatening arrest unless a fine is paid via UPI.',
]

print(f'Loading Scammer ({SCAMMER_BASE} + {SCAMMER_ADAPTER}) in bf16...')
scammer_tok = AutoTokenizer.from_pretrained(SCAMMER_BASE, trust_remote_code=True)
if scammer_tok.pad_token_id is None:
    scammer_tok.pad_token_id = scammer_tok.eos_token_id
scammer_tok.padding_side = 'left'

scammer_base = AutoModelForCausalLM.from_pretrained(
    SCAMMER_BASE, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
scammer = PeftModel.from_pretrained(scammer_base, SCAMMER_ADAPTER)
scammer.eval()
print('Scammer ready.\n')

reps_per_seed = (N_TRAINING_SCAMS + len(SEEDS) - 1) // len(SEEDS)  # round up so we get >= N_TRAINING_SCAMS
print(f'Generating {reps_per_seed} candidates x {len(SEEDS)} seeds = {reps_per_seed * len(SEEDS)} raw scams...')

REFUSAL_PHRASES = ('cannot help', 'i cannot', "i can't", 'inappropriate', 'unethical',
                   "i won't", 'against my', 'as an ai', 'illegal', 'not appropriate')

t0 = time.time()
training_scams = []
for i, seed_text in enumerate(SEEDS, 1):
    msgs = [{'role': 'system', 'content': SCAMMER_SYSTEM_PROMPT},
            {'role': 'user', 'content': seed_text}]
    text = scammer_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = scammer_tok([text] * reps_per_seed, return_tensors='pt', padding=True).to(scammer.device)
    with torch.inference_mode():
        out = scammer.generate(
            **inputs, max_new_tokens=200, do_sample=True, temperature=0.9, top_p=0.95,
            pad_token_id=scammer_tok.pad_token_id,
        )
    cands = scammer_tok.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    kept = 0
    for c in cands:
        c = c.strip()
        if 30 <= len(c) <= 600 and not any(p in c.lower()[:200] for p in REFUSAL_PHRASES):
            training_scams.append(c)
            kept += 1
    print(f'  [{i:2d}/{len(SEEDS)}]  kept {kept}/{reps_per_seed}  ({time.time()-t0:.0f}s)')

# Shuffle for diversity, trim to target size
import random
random.seed(SEED)
random.shuffle(training_scams)
training_scams = training_scams[:N_TRAINING_SCAMS]
print(f'\nFinal training set: {len(training_scams)} scams. Generation took {time.time()-t0:.0f}s.')
print('Sample:', repr(training_scams[0][:140]))

# CRITICAL: free Scammer fully before loading 7B Analyzer
del scammer, scammer_base, scammer_tok
gc.collect(); torch.cuda.empty_cache()
print('\nScammer released from VRAM.')
import subprocess
print(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader']).decode().strip())


In [ ]:
# CELL 6: Build training dataset (conversational format) + reward function with safety
import re, json
from datasets import Dataset

# EXACT prompt format the v2 Analyzer was trained on (per MODEL_CARD.md)
ANALYZER_SYSTEM_PROMPT = (
    "You are Chakravyuh's Behavioral Analyzer, a fraud detection AI deployed "
    "on-device to monitor real-time chat for Indian UPI scam patterns. "
    "Output strict JSON with `score` in [0,1], `signals` from the taxonomy, "
    "and `explanation`."
)

def build_user_prompt(scam_text):
    return f"Scammer messages:\n{scam_text}\n\nAnalyze these messages. Return strict JSON only."

# trl 0.14 conversational format -> chat template applied automatically
rows = [{'prompt': [
    {'role': 'system', 'content': ANALYZER_SYSTEM_PROMPT},
    {'role': 'user',   'content': build_user_prompt(scam)},
]} for scam in training_scams]
ds = Dataset.from_list(rows)
print(f'Built training dataset: {len(ds)} prompts (all-scams; gold score = 1.0).')


# --- Reward function ---
def _completion_to_text(raw):
    """trl 0.14 conversational mode: completions arrive as list of message dicts."""
    if isinstance(raw, str):
        return raw
    if isinstance(raw, list) and raw:
        for msg in reversed(raw):
            if isinstance(msg, dict) and msg.get('role') == 'assistant' and 'content' in msg:
                return str(msg['content'])
        last = raw[-1]
        if isinstance(last, dict) and 'content' in last:
            return str(last['content'])
    if isinstance(raw, dict) and 'content' in raw:
        return str(raw['content'])
    return str(raw)


def _parse_score(text):
    """Extract `score` value from analyzer JSON. Returns None on failure."""
    m = re.search(r'"score"\s*:\s*([\d.]+)', text)
    if m:
        try:
            v = float(m.group(1))
            if 0.0 <= v <= 1.0:
                return v
        except ValueError:
            pass
    return None


def analyzer_reward(completions, **kwargs):
    """All training inputs are SCAMS (gold score = 1.0).
    Reward = parsed_score - 0.5 (range [-0.5, +0.5]).
    Format violations (no parseable score) get -1.0 to keep the model in JSON shape."""
    rewards = []
    for raw in completions:
        text = _completion_to_text(raw)
        score = _parse_score(text)
        if score is None:
            rewards.append(-1.0)
        else:
            rewards.append(float(score) - 0.5)
    return rewards


# Smoke test the reward function with synthetic completions covering BOTH shapes
test = [
    '{"score": 0.95, "signals": ["urgency", "info_request"], "explanation": "OTP-theft scam"}',
    '{"score": 0.10, "signals": [], "explanation": "Looks legitimate"}',
    'I think this is suspicious but cannot determine the exact score.',  # JSON parse fails
    [{'role': 'assistant', 'content': '{"score": 0.88, "signals": ["impersonation"], "explanation": "fake bank"}'}],
]
print('\nReward smoke test:')
for t, r in zip(test, analyzer_reward(test)):
    preview = _completion_to_text(t)[:80]
    print(f'  reward={r:+.3f}  text={preview!r}')


In [ ]:
# CELL 7: Load Analyzer (Qwen2.5-7B 4-bit + v2 LoRA from HF Hub) — TRAINABLE for continued GRPO
import torch, trl
assert tuple(int(x) for x in trl.__version__.split('.')[:2]) >= (0, 13), \
    f'Need trl>=0.13; got {trl.__version__}.'

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

print(f'Loading Analyzer base ({ANALYZER_BASE}) in 4-bit nf4 (~5 GB VRAM)...')
analyzer_tok = AutoTokenizer.from_pretrained(ANALYZER_BASE, trust_remote_code=True)
if analyzer_tok.pad_token_id is None:
    analyzer_tok.pad_token_id = analyzer_tok.eos_token_id
analyzer_tok.padding_side = 'left'  # GRPO uses generation, needs left-padding

base = AutoModelForCausalLM.from_pretrained(
    ANALYZER_BASE,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base = prepare_model_for_kbit_training(base)

print(f'Applying v2 LoRA from HF Hub ({ANALYZER_V2_ADAPTER}) as TRAINABLE warm-start...')
# is_trainable=True so the v2 LoRA weights continue to update during phase 2 GRPO
analyzer = PeftModel.from_pretrained(base, ANALYZER_V2_ADAPTER, is_trainable=True)
analyzer.print_trainable_parameters()

# Sanity check: prompt fits inside max_prompt_length budget
sample_prompt = analyzer_tok.apply_chat_template(
    [{'role': 'system', 'content': ANALYZER_SYSTEM_PROMPT},
     {'role': 'user',   'content': build_user_prompt(training_scams[0])}],
    tokenize=True, add_generation_prompt=True,
)
print(f'\nSample chat-templated prompt length: {len(sample_prompt)} tokens (must be <= 512).')

# KL semantics note: trl 0.14 GRPO with peft uses adapter-toggling for the reference policy,
# meaning the reference is the BASE Qwen model (adapter disabled), NOT v2. With KL_BETA=0.04
# and only ~50 optimizer steps, drift from v2 will be small and the task reward dominates.
print('\n[note] trl 0.14 KL ref = base Qwen (peft adapter disabled). KL_BETA=0.04 + short training keeps drift small.')


In [ ]:
# CELL 8: GRPOConfig + safety callback (early-stop on negative-reward collapse)
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

# trl 0.14 GRPOConfig does NOT support top_p (added in 0.15). Diversity from temperature alone.
grpo_config = GRPOConfig(
    output_dir=OUTPUT_LORA_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=2,
    save_strategy='no',
    seed=SEED,
    max_prompt_length=512,        # analyzer prompt + scam text fits in ~250; 512 leaves headroom
    max_completion_length=160,    # analyzer JSON output is short
    num_generations=N_GENERATIONS,
    temperature=0.7,              # lower than phase 1 — focused JSON outputs, not diverse prose
    beta=KL_BETA,
    report_to='none',
)


class SafetyEarlyStop(TrainerCallback):
    """Halts training if mean reward stays below `neg_threshold` for `patience` consecutive
    logging windows — signal of GRPO collapse / mode degeneration."""
    def __init__(self, neg_threshold=-0.3, patience=5):
        self.neg_threshold = neg_threshold
        self.patience = patience
        self.bad_windows = 0
        self.history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        r = logs.get('reward', None)
        if r is None:
            return
        self.history.append(r)
        if r < self.neg_threshold:
            self.bad_windows += 1
            print(f'  [SafetyEarlyStop] negative reward window {self.bad_windows}/{self.patience} (r={r:+.3f})')
        else:
            self.bad_windows = 0
        if self.bad_windows >= self.patience:
            print(f'\n  [SafetyEarlyStop] TRAINING HALTED — reward < {self.neg_threshold} for {self.patience} windows.')
            print(f'  Last {self.patience} rewards: {[round(x,3) for x in self.history[-self.patience:]]}')
            print('  Skip to Cell 12 (SFT-curriculum fallback).')
            control.should_training_stop = True


safety_cb = SafetyEarlyStop(neg_threshold=-0.3, patience=5)

trainer = GRPOTrainer(
    model=analyzer,
    reward_funcs=[analyzer_reward],
    args=grpo_config,
    train_dataset=ds,
    processing_class=analyzer_tok,
    callbacks=[safety_cb],
)
print(f'GRPOTrainer ready. trl={trl.__version__}')
print(f'  Total prompts:        {len(ds)}')
print(f'  Effective batch:      {PER_DEVICE_BATCH * GRAD_ACCUM}')
print(f'  Approx opt steps:     {len(ds) // (PER_DEVICE_BATCH * GRAD_ACCUM)}')
print(f'  KL beta: {KL_BETA}, num_generations: {N_GENERATIONS}, temperature: 0.7')


In [ ]:
# CELL 9: TRAIN — phase 2 GRPO retrain (~75-90 min on A100, ~18-20 units)
# Watch the 'reward' column in trainer logs. Healthy training: reward starts ~0, climbs toward +0.3 to +0.45.
# If reward stays near 0 or goes negative, SafetyEarlyStop will halt and point you to Cell 12 (SFT fallback).
import time, torch

print('Starting phase 2 training. Reward should climb from ~0 toward +0.3-+0.5.\n')
t0 = time.time()
training_succeeded = False
try:
    trainer.train()
    training_succeeded = (safety_cb.bad_windows < safety_cb.patience)
    if training_succeeded:
        print(f'\n✓ Training complete in {(time.time()-t0)/60:.1f} min.')
        if safety_cb.history:
            recent = safety_cb.history[-5:]
            print(f'  Last 5 reward values: {[round(x,3) for x in recent]}')
            print(f'  Final mean reward: {sum(recent)/len(recent):+.3f}')
    else:
        print(f'\n✗ Training halted by safety callback. Skip to Cell 12 (SFT fallback).')
except torch.cuda.OutOfMemoryError:
    print('\n✗ CUDA OOM. Reduce in Cell 8 and re-run from Cell 7:')
    print('  - num_generations=2  (fastest fix; halves memory)')
    print('  - max_completion_length=120  (smaller activation footprint)')
    print('  - per_device_train_batch_size=1 + gradient_accumulation_steps=8')
    raise
except Exception as e:
    print(f'\n✗ Training failed: {type(e).__name__}: {e}')
    print('Use Cell 12 (SFT-curriculum fallback) for the guaranteed-success backup.')
    raise


In [ ]:
# CELL 10: Save the v2-coevolved LoRA + bundle for download
import shutil
from pathlib import Path

if training_succeeded:
    final_dir = f'{OUTPUT_LORA_DIR}/final'
    trainer.save_model(final_dir)
    print(f'Saved v2-coevolved LoRA to: {final_dir}')
    !ls -la {final_dir}/
    bundle = '/content/analyzer_lora_v2_coevolved.zip'
    shutil.make_archive(bundle.replace('.zip', ''), 'zip', final_dir)
    print(f'\nBundle: {bundle} ({Path(bundle).stat().st_size / 1e6:.1f} MB)')
else:
    print('Training did not succeed — skip Cell 11 and run Cell 12 (SFT fallback) instead.')


In [ ]:
# CELL 11: Re-evaluate — score the SAME 64 best-of-8 Scammer outputs with the new v2-coevolved Analyzer
# This gives a clean before/after vs phase 1 head-to-head (33% bypass).
import json, time
from pathlib import Path

if not training_succeeded:
    print('Skipping eval — training did not succeed. Run Cell 12 (SFT fallback).')
else:
    phase1 = json.load(open(PHASE1_EVAL_PATH))
    samples = phase1['samples']
    print(f'Loaded {len(samples)} phase-1 best-of-8 samples for re-evaluation.\n')

    # The trainer.model still has the trained LoRA loaded — score directly without reload
    coevolved = trainer.model
    coevolved.eval()

    def coevolved_score(text):
        msgs = [{'role': 'system', 'content': ANALYZER_SYSTEM_PROMPT},
                {'role': 'user',   'content': build_user_prompt(text)}]
        prompt = analyzer_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = analyzer_tok(prompt, return_tensors='pt').to(coevolved.device)
        with torch.inference_mode():
            out = coevolved.generate(**inputs, max_new_tokens=160, do_sample=False,
                                      pad_token_id=analyzer_tok.pad_token_id)
        resp = analyzer_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        score = _parse_score(resp)
        return (score if score is not None else 0.5, resp.strip()[:300])

    print('Scoring 64 samples with v2-coevolved Analyzer...')
    t0 = time.time()
    parse_failures = 0
    for i, s in enumerate(samples, 1):
        score, raw = coevolved_score(s['completion'])
        s['v2_coevolved_score'] = round(score, 3)
        s['v2_coevolved_caught'] = score >= 0.5
        s['v2_coevolved_bypass'] = not s['v2_coevolved_caught']
        if score == 0.5 and '"score"' not in raw:
            parse_failures += 1
        if i % 8 == 0 or i == len(samples):
            print(f'  [{i:3d}/{len(samples)}]  ({time.time()-t0:.0f}s)')
    print(f'\nDone. JSON parse failures: {parse_failures}/{len(samples)}\n')

    # Aggregate
    def wilson_ci(k, n, z=1.96):
        if n == 0: return (0.0, 0.0)
        p = k/n; denom = 1 + z*z/n
        center = (p + z*z/(2*n)) / denom
        margin = z * ((p*(1-p)/n + z*z/(4*n*n))**0.5) / denom
        return (max(0.0, center-margin), min(1.0, center+margin))

    def summarize(samples_subset, label):
        n = len(samples_subset)
        if n == 0: return {'label': label, 'n': 0}
        v2_byp = sum(s['v2_bypass'] for s in samples_subset)
        co_byp = sum(s['v2_coevolved_bypass'] for s in samples_subset)
        v_lo, v_hi = wilson_ci(v2_byp, n)
        c_lo, c_hi = wilson_ci(co_byp, n)
        return {
            'label': label, 'n': n,
            'v2_bypass_count': v2_byp, 'v2_bypass_rate': v2_byp / n,
            'v2_wilson_95_ci': [round(v_lo, 3), round(v_hi, 3)],
            'coevolved_bypass_count': co_byp, 'coevolved_bypass_rate': co_byp / n,
            'coevolved_wilson_95_ci': [round(c_lo, 3), round(c_hi, 3)],
            'reduction_pp': round((v2_byp - co_byp) / n * 100, 1),
        }

    overall = summarize(samples, 'overall_n64')
    train_  = summarize([s for s in samples if s['split']=='train'],    'train_n32')
    heldout = summarize([s for s in samples if s['split']=='held_out'], 'held_out_n32')

    print('=' * 110)
    print('CO-EVOLUTION RESULT  (round 1: v2 vs Scammer  ->  round 2: v2-coevolved vs same Scammer)')
    print('=' * 110)
    print(f"  {'Split':<14}  {'n':>4}  {'v2 (round 1)':>20}  {'v2-coevolved (round 2)':>26}  {'Reduction (pp)':>15}")
    print('  ' + '-'*108)
    for r in (overall, train_, heldout):
        v_pct = f"{r['v2_bypass_count']}/{r['n']} = {r['v2_bypass_rate']:.1%}"
        c_pct = f"{r['coevolved_bypass_count']}/{r['n']} = {r['coevolved_bypass_rate']:.1%}"
        print(f"  {r['label']:<14}  {r['n']:>4}  {v_pct:>20}  {c_pct:>26}  {r['reduction_pp']:>+13.1f}")

    # Capture training trajectory (used by eval/plot_coevolution.py for the reward-curve plot)
    log_history = []
    try:
        for entry in trainer.state.log_history:
            keep = {k: v for k, v in entry.items()
                    if k in ('step', 'epoch', 'reward', 'reward_std', 'kl', 'loss', 'learning_rate')}
            if keep:
                log_history.append(keep)
    except Exception as e:
        print(f'  [warn] could not capture log_history: {e}')

    LOGS = Path(REPO_DIR) / 'logs'
    LOGS.mkdir(exist_ok=True)
    out_path = LOGS / 'b2_phase2_coevolution_eval.json'
    out_path.write_text(json.dumps({
        'meta': {
            'analyzer_base': ANALYZER_BASE,
            'analyzer_warm_start': ANALYZER_V2_ADAPTER,
            'phase2_lora_local': f'{OUTPUT_LORA_DIR}/final',
            'phase2_training': {
                'method': 'GRPO continued training',
                'n_training_scams': len(training_scams),
                'epochs': NUM_TRAIN_EPOCHS,
                'lr': LR,
                'kl_beta': KL_BETA,
                'num_generations': N_GENERATIONS,
                'effective_batch': PER_DEVICE_BATCH * GRAD_ACCUM,
                'temperature': 0.7,
                'trl_version': trl.__version__,
                'kl_reference_note': 'trl 0.14 peft adapter-toggle => KL ref is base Qwen, not v2. Documented limitation.',
            },
            'training_log_history': log_history,   # consumed by eval/plot_coevolution.py
            'eval_design': (
                'Same 64 best-of-8 Scammer outputs from b2_phase1_scammer_eval_n64_bestof8.json, '
                'rescored by both the round-1 v2 Analyzer (cached scores) and the round-2 v2-coevolved Analyzer.'
            ),
            'parse_failures': parse_failures,
        },
        'aggregate': {'overall': overall, 'train_seeds': train_, 'held_out_seeds': heldout},
        'samples': samples,
    }, indent=2))
    print(f'\nSaved: {out_path}')

    try:
        from google.colab import files
        files.download(str(out_path))
        files.download('/content/analyzer_lora_v2_coevolved.zip')
    except ImportError:
        print('Not in Colab — files at:'); print(' ', out_path); print('  /content/analyzer_lora_v2_coevolved.zip')


In [ ]:
# CELL 12: [FALLBACK ONLY] SFT-curriculum — run THIS instead of Cells 9-11 if GRPO collapsed/OOMed
# Uses the same 200 Scammer-generated scams as supervised training data with heuristic gold JSON.
# ~25-30 min on A100, ~5 units, basically guaranteed to converge (no RL instability).
import torch, gc, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Free anything from Cell 7-9 (might still be in memory)
try:
    del trainer
except NameError:
    pass
try:
    del analyzer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

# Build (input, target) pairs — heuristic gold JSON for each generated scam
def gold_json(scam_text):
    """Heuristic gold: high score + signals chosen by simple keyword matches."""
    text_l = scam_text.lower()
    signals = []
    if any(k in text_l for k in ['otp', 'pin', 'cvv', 'aadhaar', 'kyc', 'verify', 'confirm', 'share']):
        signals.append('info_request')
    if any(k in text_l for k in ['urgent', 'immediately', 'expire', 'block', 'suspended', 'last chance', '24 hour', 'now']):
        signals.append('urgency')
    if any(k in text_l for k in ['sbi', 'hdfc', 'icici', 'axis', 'rbi', 'income tax', 'epf', 'paytm', 'amazon', 'flipkart']):
        signals.append('impersonation')
    if any(k in text_l for k in ['http', 'click here', 'link', 'bit.ly', '.in/', '.com/', 'tinyurl']):
        signals.append('suspicious_link')
    if any(k in text_l for k in ['win', 'lottery', 'prize', 'reward', 'lakh', 'crore', 'free']):
        signals.append('greed')
    if not signals:
        signals = ['info_request']
    return json.dumps({
        'score': 0.95,
        'signals': signals,
        'explanation': 'Likely UPI scam; matches Indian fraud patterns.',
    })

sft_rows = [{'messages': [
    {'role': 'system',    'content': ANALYZER_SYSTEM_PROMPT},
    {'role': 'user',      'content': build_user_prompt(scam)},
    {'role': 'assistant', 'content': gold_json(scam)},
]} for scam in training_scams]
sft_ds = Dataset.from_list(sft_rows)
print(f'SFT dataset: {len(sft_ds)} (scam, gold-JSON) pairs.')

# Reload Analyzer fresh
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
analyzer_tok = AutoTokenizer.from_pretrained(ANALYZER_BASE, trust_remote_code=True)
if analyzer_tok.pad_token_id is None:
    analyzer_tok.pad_token_id = analyzer_tok.eos_token_id
base = AutoModelForCausalLM.from_pretrained(ANALYZER_BASE, quantization_config=bnb_cfg,
                                            device_map='auto', trust_remote_code=True,
                                            torch_dtype=torch.bfloat16)
base = prepare_model_for_kbit_training(base)
analyzer = PeftModel.from_pretrained(base, ANALYZER_V2_ADAPTER, is_trainable=True)
analyzer.print_trainable_parameters()

OUTPUT_LORA_DIR_SFT = OUTPUT_LORA_DIR + '_sft'
sft_config = SFTConfig(
    output_dir=OUTPUT_LORA_DIR_SFT,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=5,
    save_strategy='no',
    seed=SEED,
    max_seq_length=768,
    report_to='none',
)
sft_trainer = SFTTrainer(
    model=analyzer,
    args=sft_config,
    train_dataset=sft_ds,
    processing_class=analyzer_tok,
)
print('Training (SFT)... ~25-30 min on A100.')
sft_trainer.train()
final_dir_sft = OUTPUT_LORA_DIR_SFT + '/final'
sft_trainer.save_model(final_dir_sft)
print(f'\nSaved SFT-curriculum LoRA to: {final_dir_sft}')

# Bundle for download
import shutil
from pathlib import Path
bundle_sft = '/content/analyzer_lora_v2_coevolved_sft.zip'
shutil.make_archive(bundle_sft.replace('.zip', ''), 'zip', final_dir_sft)
print(f'Bundle: {bundle_sft} ({Path(bundle_sft).stat().st_size / 1e6:.1f} MB)')

# Alias for Cell 11 to work transparently if you re-run it now
trainer = sft_trainer
training_succeeded = True
OUTPUT_LORA_DIR = OUTPUT_LORA_DIR_SFT
print('\n' + '=' * 60)
print('SFT fallback complete. Now re-run Cell 11 to evaluate this adapter on the same 64 samples.')
print('Cell 11 will write logs/b2_phase2_coevolution_eval.json with the new bypass numbers.')
print('=' * 60)


## Next steps after Phase 2 finishes

You should have downloaded two artifacts: `b2_phase2_coevolution_eval.json` and `analyzer_lora_v2_coevolved.zip` (or `_sft.zip` if you fell back). On your laptop, from `/home/palkia/code/Chakravyuh/`:

```bash
# 1. Move artifacts into the repo
mv ~/Downloads/b2_phase2_coevolution_eval.json logs/
mkdir -p checkpoints/analyzer_lora_v2_coevolved
unzip ~/Downloads/analyzer_lora_v2_coevolved.zip -d checkpoints/analyzer_lora_v2_coevolved/

# 2. Push the new LoRA to HF Hub (~30 sec)
hf upload ujjwalpardeshi/chakravyuh-analyzer-lora-v2-coevolved \
  checkpoints/analyzer_lora_v2_coevolved/ . --repo-type model \
  --commit-message "Initial: B.2 phase 2 v2-coevolved Analyzer LoRA (continued GRPO vs trained Scammer)"

# 3. Regenerate co-evolution plots from new data (now includes round-2 + training reward curve)
python3 eval/plot_coevolution.py
# Writes plots/chakravyuh_plots/coevolution_*.png — drop into slides directly.

# 4. Commit eval JSON + new plots (zip is gitignored)
git add logs/b2_phase2_coevolution_eval.json plots/chakravyuh_plots/coevolution_*.png
git commit -m "feat: B.2 phase 2 SHIPPED — co-evolution round 2 result + plots"
git push
```

Then the README/slide writes itself:

> **Round 1.** v2 Analyzer LoRA vs trained Scammer = **33% bypass** rate (n=64).
> **Round 2.** v2-coevolved Analyzer LoRA (continued GRPO vs same Scammer) = **X% bypass** rate.
> Both adapters live on HF Hub. Anyone can reproduce the gap in 5 lines of code.

Plots produced (already in `plots/chakravyuh_plots/`):
- `coevolution_headline.png` — 3-bar comparison (Scripted / v2 / v2-coevolved) by split
- `coevolution_per_category.png` — per-seed bypass before/after
- `coevolution_score_movement.png` — per-sample scatter (v2 score vs v2-coevolved score)
- `coevolution_training_reward.png` — phase 2 GRPO reward curve over steps
- `scammer_phase1_per_category.png` — Scammer-side context (single-shot vs best-of-8)
